# Differentiable Fluoddity — gecko silhouette

Trains the [Fluoddity-Core](https://github.com/aphid91/Fluoddity-Core) particle system (ported to PyTorch in `diffluoditty/fluoddity.py`) with the same pool / damage / BPTT recipe as the NCA notebook.
The trainable parameters are the rule (10 Fourier centres = 80 numbers) and the physics sliders, per config, plus the two global trail settings.

Particles start in a small dense disc at the centre, all heading the same way (that breaks the system's rotation/mirror symmetry so the gecko has a definite pose). The loss compares particle **density** (rendered as a black silhouette) to the gecko's shape.

**Runtime → Change runtime type → GPU.**

In [ ]:
import os
if os.path.exists("../diffluoditty/__init__.py"):
    %cd ..
elif not os.path.exists("diffluoditty/__init__.py"):
    if not os.path.exists("diffluoditty"):
        !git clone -q https://github.com/aphid91/diffluoditty.git
    %cd diffluoditty
!git pull -q
%load_ext autoreload
%autoreload 2

In [ ]:
import glob, math
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import HTML

from diffluoditty import (Fluoddity, Trainer, experiment_config, load_emoji, pad_target, silhouette,
                          circle_masks, half_mask, evaluate, load_runs, gradient_horizon, plot_gradient_horizon)
from diffluoditty import viz

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name() if device.type == "cuda" else "")

## Target

In [ ]:
target = silhouette(pad_target(load_emoji("gecko", max_size=40), padding=16))
viz.show_images(viz.to_rgb(target)[None], scale=3)

## System
`INIT` picks the starting rule and physics: a Fluoddity-Core config from `configs/` (its cohorts become configs, with Core's mutation applied once at load time), or `random` (Core's random rule generator; physics from 9LeafClovers).
`N_CONFIGS` = how many independent rule+physics sets the particles are split between (0 = the config file's cohort count).
`ALPHA_DENSITY` = particles per render pixel that give alpha ≈ 0.63; 0 = auto (spreading every particle evenly over the gecko gives alpha ≈ 0.86).

**The `DETACH_*` switches only change the backward pass** (the simulation itself is identical). Individual particle paths are chaotic: exact gradients grow ~5-20x *per step* and are noise after ~4 steps. Treating where particles sense, which way they face, and where they deposit as constants in the backward pass cuts those feedback loops: the gradient becomes biased, but stays stable and consistent over 64+ steps. Run the horizon diagnostic below with them on and off to see the difference.

In [ ]:
#@title System { run: "auto" }
INIT = "9LeafClovers"  #@param ["random", "9LeafClovers", "Bramble", "Bubbles", "Cilia", "Circuits", "Corners", "HungryHungryHippos", "Meandering", "Nettle", "Pop", "ReactionConfusion", "Sidewinder", "Vacuoles", "Web"]
N_CONFIGS = 1  #@param {type:"integer"}
RULE_SEED = 0.5  #@param {type:"number"}
N_PARTICLES = 20000  #@param {type:"integer"}
CANVAS_SIZE = 192  #@param {type:"integer"}
SEED_RADIUS = 0.05  #@param {type:"number"}
SEED_SPEED_PX = 0.5  #@param {type:"number"}
ALPHA_DENSITY = 0  #@param {type:"number"}
DETACH_SENSING = True  #@param {type:"boolean"}
DETACH_HEADING = True  #@param {type:"boolean"}
DETACH_DEPOSIT = True  #@param {type:"boolean"}

system_kwargs = dict(
    n_particles=N_PARTICLES, canvas_size=CANVAS_SIZE, render_size=target.shape[-1],
    seed_radius=SEED_RADIUS, seed_speed_px=SEED_SPEED_PX,
    alpha_density=ALPHA_DENSITY or N_PARTICLES / target[3].sum().item() / 2,
    detach_sensing=DETACH_SENSING, detach_heading=DETACH_HEADING, detach_deposit=DETACH_DEPOSIT,
)

def make_system():
    if INIT == "random":
        return Fluoddity(n_configs=max(1, N_CONFIGS), rule_seed=RULE_SEED, **system_kwargs)
    return Fluoddity.from_core_config(f"configs/{INIT}.json", n_configs=N_CONFIGS or None, **system_kwargs)

flu = make_system().to(device)
n_params = sum(p.numel() for p in flu.parameters())
print(f"{INIT}: {flu.n_configs} config(s), {N_PARTICLES:,} particles, canvas {CANVAS_SIZE}^2 "
      f"({N_PARTICLES / CANVAS_SIZE**2:.2f} particles/px; Core runs at ~0.57)")
print(f"{n_params} trainable parameters = {flu.n_configs} x (80 rule + 8 physics) + 2 global")
{k: [round(x, 4) for x in v.reshape(-1).tolist()] for k, v in flu.physics_values().items()}

### What does it do before training?
The seed disc, rolled forward without gradients. Left: particle density (what the loss sees). Right: trail field speed.

In [ ]:
def preview(system, n_steps=400, every=4):
    with torch.no_grad():
        x = system.seed(1)
        frames = []
        for t in range(n_steps + 1):
            if t % every == 0:
                dens = viz.to_rgb(system.render(x))[0]
                trails = x["canvas"][0].norm(dim=0)
                trails = (trails / (trails.max() + 1e-8)).sqrt().cpu().numpy()
                trails = torch.nn.functional.interpolate(torch.tensor(trails)[None, None], size=dens.shape[:2])[0, 0].numpy()
                frames.append(np.concatenate([dens, plt.cm.magma(trails)[..., :3]], axis=1))
            x = system.step(x)
    return frames

HTML(viz.animate(preview(flu), scale=3).to_jshtml())

## How far back can gradients reach?
Gradient size and seed-to-seed agreement against backprop horizon (see `diffluoditty/diagnostics.py`). If agreement collapses towards 0 at horizon T, training windows much longer than T are mostly noise — keep `MAX_STEPS` below that.
Measured on CPU for 9LeafClovers (20k particles, after 50 warm-up steps): exact gradients reach |grad| ~ 1e6 by T=12 with agreement < 0; with all three `DETACH_*` on, agreement stays 0.93–0.99 out to T=64.

In [ ]:
horizon = gradient_horizon(flu, target, horizons=(2, 4, 8, 16, 32, 64), batch_size=4, n_batches=3, warmup=50, grad_checkpoint_every=8)
for r in horizon:
    print(f"T={r['horizon']:4d}  loss {r['loss']:.4f}  |grad| {r['grad_norm']:.3e}  agreement {r['agreement']:+.3f}")
plot_gradient_horizon(horizon)

## Training settings
Same recipe as the NCA. `GRAD_CHECKPOINT_EVERY` trades compute for memory: 20k particles × 96 steps × batch 8 doesn't fit in a T4 without it.
`LR` is in "slider units" — physics parameters are stored as value / slider-span, so one learning rate suits all of them.

In [ ]:
#@title Training { run: "auto" }
EXPERIMENT = "regenerating"  #@param ["growing", "persistent", "regenerating"]
N_ITERS = 4000  #@param {type:"integer"}
LR = 5e-3  #@param {type:"number"}
LR_DECAY_AT = 2000  #@param {type:"integer"}
BATCH_SIZE = 8  #@param {type:"integer"}
POOL_SIZE = 256  #@param {type:"integer"}
MIN_STEPS = 32  #@param {type:"integer"}
MAX_STEPS = 64  #@param {type:"integer"}
BURN_IN = 0  #@param {type:"integer"}
#@markdown `BURN_IN`: extra steps run without gradients before each window (moves pool states along faster).
GRAD_CHECKPOINT_EVERY = 8  #@param {type:"integer"}
SEED = 0  #@param {type:"integer"}

cfg = experiment_config(EXPERIMENT, lr=LR, lr_milestones=[LR_DECAY_AT], batch_size=BATCH_SIZE, pool_size=POOL_SIZE,
                        min_steps=MIN_STEPS, max_steps=MAX_STEPS, burn_in=BURN_IN, grad_checkpoint_every=GRAD_CHECKPOINT_EVERY, log_every=20)
RUN_NAME = f"flu_{EXPERIMENT}_{INIT}_k{flu.n_configs}_n{N_PARTICLES}_lr{LR:g}_s{SEED}"
print("run:", RUN_NAME)
cfg

Running this cell **starts a fresh model** (re-creates the system from the System form).

In [ ]:
torch.manual_seed(SEED)
flu = make_system().to(device)
trainer = Trainer(flu, target, cfg)
print(f"{RUN_NAME}: {sum(p.numel() for p in flu.parameters())} parameters")

## Train
Interrupt whenever you like; re-running continues where it stopped.

In [ ]:
try:
    trainer.train(N_ITERS - trainer.step_count, callback=viz.show_progress, callback_every=20)
except KeyboardInterrupt:
    print(f"interrupted at step {trainer.step_count}")

## Evaluate and save

In [ ]:
result = evaluate(flu, target, n=8, persist_steps=600, damage_at=200, regen_steps=200)
print(f"{RUN_NAME}  ({sum(p.numel() for p in flu.parameters())} params)")
for k in ["grow_loss", "persist_loss", "regen_loss", "died", "exploded"]:
    print(f"  {k:13s} {result[k]:.4f}")
os.makedirs("checkpoints", exist_ok=True)
trainer.save(f"checkpoints/{RUN_NAME}.pt", run_name=RUN_NAME, eval=result)
# Later: flu = Fluoddity.from_checkpoint(f"checkpoints/{RUN_NAME}.pt", device)

## Training diagnostics

In [ ]:
h = trainer.history
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
viz.plot_history(h, axes[0])
axes[1].semilogy([r["grad_norm"] for r in h], ".", markersize=1, alpha=0.3)
axes[1].set_title(f"raw gradient norm ({sum(r.get('skipped', False) for r in h)} non-finite updates skipped)")
axes[2].plot([r["seconds"] for r in h]); axes[2].set_title("seconds / iteration")
plt.show()

# How the physics moved during training (start -> now)
start = make_system().physics_values()
for k, v in flu.physics_values().items():
    print(f"{k:18s} {[round(x, 3) for x in start[k].reshape(-1).tolist()]} -> {[round(x, 3) for x in v.reshape(-1).tolist()]}")

In [ ]:
if trainer.pool is not None:
    idx = torch.randperm(cfg.pool_size)[:32]
    viz.show_images(viz.to_rgb(flu.render({k: v[idx] for k, v in trainer.pool.state.items()})), ncols=8)

## Growth from the seed

In [ ]:
HTML(viz.animate(preview(flu, n_steps=600, every=4), scale=3).to_jshtml())

## Regeneration
Grow for 200 steps, cut four copies in different ways, watch them heal.

In [ ]:
h_, w_ = target.shape[-2:]
cuts = torch.cat([half_mask(1, h_, w_, "right"), half_mask(1, h_, w_, "top"), circle_masks(2, h_, w_)]).to(device)
frames = viz.record(flu, flu.seed(4), n_steps=500, every=4, events={200: lambda s: flu.damage(s, cuts)})
HTML(viz.animate(frames, scale=2).to_jshtml())

## Compare runs
Every checkpoint from this session (NCA runs included, if any).

In [ ]:
import pandas as pd
runs = load_runs("checkpoints/*.pt", target, device)
display(pd.DataFrame([
    {"run": r["name"], "params": r["n_params"], **{k: r["eval"][k] for k in ["grow_loss", "persist_loss", "regen_loss", "died", "exploded"]}}
    for r in runs
]).set_index("run").style.format(precision=4))
viz.compare_runs(runs)